In [6]:
# 소비자 cli interface

import sqlite3
import uuid
from datetime import date
from pathlib import Path

DB_PATH = Path("../schema/파리바게트.db")


def get_connection():
    if not DB_PATH.exists():
        raise FileNotFoundError(f"DB 파일을 찾을 수 없습니다: {DB_PATH}")
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON;")
    return conn


def generate_customer_id(conn):
    while True:
        cid = "C" + uuid.uuid4().hex[:8].upper()
        row = conn.execute("SELECT 1 FROM 고객 WHERE 고객ID=?", (cid,)).fetchone()
        if not row:
            return cid


def generate_sale_id(conn):
    while True:
        sid = "S" + uuid.uuid4().hex[:8].upper()
        row = conn.execute("SELECT 1 FROM 판매 WHERE 판매번호=?", (sid,)).fetchone()
        if not row:
            return sid


# 로그인
def login(conn):
    print("\n── 회원 로그인 ──")
    username = input("아이디: ").strip()
    password = input("비밀번호: ").strip()

    row = conn.execute(
        """SELECT 회원.고객ID, 고객.이름, 회원.아이디
           FROM 회원 JOIN 고객 ON 회원.고객ID=고객.고객ID
           WHERE 회원.아이디=? AND 회원.비밀번호=?""",
        (username, password)
    ).fetchone()

    if not row:
        print("아이디 또는 비밀번호가 올바르지 않습니다.")
        return None
    print(f"\n환영합니다, {row['이름']}님!")
    return {"고객ID": row["고객ID"], "이름": row["이름"], "아이디": row["아이디"], "회원여부": True}


# 비회원 진입
def enter_as_nonmember(conn):
    print("\n비회원으로 진입합니다.")
    cid = generate_customer_id(conn)
    try:
        conn.execute("BEGIN")
        conn.execute("INSERT INTO 고객 (고객ID, 이름) VALUES (?, ?)", (cid, "비회원"))
        conn.execute("INSERT INTO 비회원 (고객ID, 임시식별여부) VALUES (?, ?)", (cid, 1))
        conn.commit()
    except sqlite3.Error as e:
        conn.rollback()
        print(f"오류: {e}")
        return None
    return {"고객ID": cid, "이름": "비회원", "회원여부": False}


# 매장 선택
def select_branch(conn):
    print("\n── 매장 선택 ──")
    rows = conn.execute(
        "SELECT 지점명, 지역_시도, 영업시간 FROM 지점 ORDER BY 지역_시도, 지점명"
    ).fetchall()

    print(f"\n{'번호':<4} {'지점명':<15} {'지역':<8} {'영업시간'}")
    print("-" * 50)
    for i, r in enumerate(rows, 1):
        print(f"{i:<4} {r['지점명']:<15} {r['지역_시도']:<8} {r['영업시간']}")

    while True:
        try:
            choice = int(input("\n매장 번호 선택: ").strip()) - 1
            if 0 <= choice < len(rows):
                branch = rows[choice]["지점명"]
                print(f"선택된 매장: {branch}")
                return branch
            print("올바른 번호를 입력하세요.")
        except ValueError:
            print("숫자를 입력하세요.")


# 상품 검색
def search_products(conn, branch):
    print("\n── 상품 검색 ──")
    print("1. 전체 보기")
    print("2. 이름으로 검색")
    print("3. 카테고리로 검색")
    print("4. 브랜드로 검색")
    choice = input("선택 > ").strip()

    base_sql = """
        SELECT 상품.상품코드, 상품.이름, 브랜드.브랜드명,
               보유.재고수량, 보유.매장가격,
               CASE
                   WHEN EXISTS (SELECT 1 FROM 빵 WHERE 빵.상품코드=상품.상품코드) THEN '빵'
                   WHEN EXISTS (SELECT 1 FROM 케이크 WHERE 케이크.상품코드=상품.상품코드) THEN '케이크'
                   WHEN EXISTS (SELECT 1 FROM 샐러드 WHERE 샐러드.상품코드=상품.상품코드) THEN '샐러드'
                   WHEN EXISTS (SELECT 1 FROM 음료 WHERE 음료.상품코드=상품.상품코드) THEN '음료'
                   WHEN EXISTS (SELECT 1 FROM 스낵 WHERE 스낵.상품코드=상품.상품코드) THEN '스낵'
                   ELSE '기타'
               END AS 카테고리
        FROM 보유
        JOIN 상품 ON 보유.상품코드=상품.상품코드
        LEFT JOIN 브랜드 ON 상품.브랜드코드=브랜드.브랜드코드
        WHERE 보유.지점명=? AND 보유.재고수량 > 0
    """

    params = [branch]

    if choice == "2":
        keyword = input("상품명 검색어: ").strip()
        base_sql += " AND 상품.이름 LIKE ?"
        params.append(f"%{keyword}%")
    elif choice == "3":
        print("빵 / 케이크 / 샐러드 / 음료 / 스낵")
        cat = input("카테고리: ").strip()
        base_sql += f"""
            AND EXISTS (
                SELECT 1 FROM {cat} WHERE {cat}.상품코드=상품.상품코드
            )
        """
    elif choice == "4":
        brands = conn.execute(
            """SELECT DISTINCT 브랜드.브랜드명
               FROM 보유
               JOIN 상품 ON 보유.상품코드=상품.상품코드
               JOIN 브랜드 ON 상품.브랜드코드=브랜드.브랜드코드
               WHERE 보유.지점명=? AND 보유.재고수량 > 0
               ORDER BY 브랜드.브랜드명""",
            (branch,)
        ).fetchall()

        if not brands:
            print("조회 가능한 브랜드가 없습니다.")
            return []

        print("\n브랜드 목록:")
        for i, b in enumerate(brands, 1):
            print(f"{i}. {b['브랜드명']}")

        while True:
            try:
                b_choice = int(input("브랜드 번호 선택: ").strip()) - 1
                if 0 <= b_choice < len(brands):
                    selected_brand = brands[b_choice]["브랜드명"]
                    break
                print("올바른 번호를 입력하세요.")
            except ValueError:
                print("숫자를 입력하세요.")

        base_sql += " AND 브랜드.브랜드명=?"
        params.append(selected_brand)

    base_sql += " ORDER BY 카테고리, 상품.이름"

    try:
        rows = conn.execute(base_sql, params).fetchall()
    except sqlite3.Error as e:
        print(f"검색 오류: {e}")
        return []

    if not rows:
        print("검색 결과가 없습니다.")
        return []

    print(f"\n{'번호':<4} {'상품코드':<8} {'이름':<20} {'카테고리':<8} {'브랜드':<12} {'가격':>8} {'재고':>6}")
    print("-" * 75)
    for i, r in enumerate(rows, 1):
        print(f"{i:<4} {r['상품코드']:<8} {r['이름']:<20} {r['카테고리']:<8} {r['브랜드명'] or '-':<12} {r['매장가격']:>8,}원 {r['재고수량']:>5}개")

    return rows


# 구매
def purchase(conn, user, branch):
    print("\n── 구매 ──")
    rows = search_products(conn, branch)
    if not rows:
        return

    cart = []
    while True:
        try:
            choice = input("\n구매할 상품 번호 (완료는 0): ").strip()
            if choice == "0":
                break
            idx = int(choice) - 1
            if not (0 <= idx < len(rows)):
                print("올바른 번호를 입력하세요.")
                continue
            product = rows[idx]
            qty = int(input(f"수량 (재고: {product['재고수량']}개): ").strip())
            if qty <= 0:
                print("1 이상 입력하세요.")
                continue
            if qty > product["재고수량"]:
                print("재고가 부족합니다.")
                continue
            cart.append({"상품코드": product["상품코드"], "이름": product["이름"],
                         "수량": qty, "단가": product["매장가격"]})
            print(f"장바구니 추가: {product['이름']} x {qty}")
        except ValueError:
            print("숫자를 입력하세요.")

    if not cart:
        print("장바구니가 비어있습니다.")
        return

    # 영수증 미리 보기
    print("\n── 주문 내역 ──")
    total = 0
    for item in cart:
        subtotal = item["수량"] * item["단가"]
        total += subtotal
        print(f"  {item['이름']} x {item['수량']} = {subtotal:,}원")
    print(f"  합계: {total:,}원")

    print("\n결제수단: 1.현금 / 2.카드 / 3.간편결제")
    pm_map = {"1": "현금", "2": "카드", "3": "간편결제"}
    pm_choice = input("선택 > ").strip()
    payment = pm_map.get(pm_choice, "카드")

    confirm = input("구매하시겠습니까? (Y/N): ").strip().upper()
    if confirm != "Y":
        print("구매를 취소했습니다.")
        return

    emp_id = None  # 소비자 CLI는 직원 개입 없음

    sale_id = generate_sale_id(conn)
    today = date.today().isoformat()

    try:
        conn.execute("BEGIN IMMEDIATE")
        # 재고 재확인
        for item in cart:
            stock = conn.execute(
                "SELECT 재고수량 FROM 보유 WHERE 지점명=? AND 상품코드=?",
                (branch, item["상품코드"])
            ).fetchone()
            if not stock or stock["재고수량"] < item["수량"]:
                conn.rollback()
                print(f"'{item['이름']}' 재고가 부족합니다. 구매를 취소합니다.")
                return

        # 판매 INSERT
        conn.execute(
            "INSERT INTO 판매 (판매번호, 판매일자, 총판매금액, 결제수단, 직원ID, 고객ID, 지점명) VALUES (?,?,?,?,?,?,?)",
            (sale_id, today, total, payment, emp_id, user["고객ID"], branch)
        )

        # 판매상세 INSERT + 재고 차감
        for j, item in enumerate(cart, 1):
            conn.execute(
                "INSERT INTO 판매상세 (판매번호, 항목번호, 수량, 상품코드, 판매단가) VALUES (?,?,?,?,?)",
                (sale_id, j, item["수량"], item["상품코드"], item["단가"])
            )
            conn.execute(
                "UPDATE 보유 SET 재고수량=재고수량-? WHERE 지점명=? AND 상품코드=?",
                (item["수량"], branch, item["상품코드"])
            )

        conn.commit()
        print(f"\n구매 완료! 판매번호: {sale_id}")
        print(f"결제금액: {total:,}원 ({payment})")

    except sqlite3.Error as e:
        conn.rollback()
        print(f"구매 실패: {e}")


# 환불 (회원만)
def refund(conn, user, sale_id):
    # 본인 구매인지 확인
    sale = conn.execute(
        "SELECT * FROM 판매 WHERE 판매번호=? AND 고객ID=?",
        (sale_id, user["고객ID"])
    ).fetchone()

    if not sale:
        print("해당 판매번호를 찾을 수 없거나 본인 구매가 아닙니다.")
        return

    # 환불 대상 상세 조회
    details = conn.execute(
        """SELECT 판매상세.상품코드, 판매상세.수량, 상품.이름, 판매.지점명
           FROM 판매상세
           JOIN 상품 ON 판매상세.상품코드=상품.상품코드
           JOIN 판매 ON 판매상세.판매번호=판매.판매번호
           WHERE 판매상세.판매번호=?""",
        (sale_id,)
    ).fetchall()

    print(f"\n── 환불 대상 내역 [{sale_id}] ──")
    for d in details:
        print(f"  {d['이름']} x {d['수량']}")
    print(f"  결제금액: {sale['총판매금액']:,}원")

    confirm = input("환불하시겠습니까? (Y/N): ").strip().upper()
    if confirm != "Y":
        print("환불을 취소했습니다.")
        return

    try:
        conn.execute("BEGIN IMMEDIATE")

        # 재고 원복
        for d in details:
            conn.execute(
                "UPDATE 보유 SET 재고수량=재고수량+? WHERE 지점명=? AND 상품코드=?",
                (d["수량"], sale["지점명"], d["상품코드"])
            )

        # 판매 삭제 (CASCADE로 판매상세 자동 삭제)
        conn.execute("DELETE FROM 판매 WHERE 판매번호=?", (sale_id,))

        conn.commit()
        print(f"환불 완료. {sale['총판매금액']:,}원이 환불되었습니다.")

    except sqlite3.Error as e:
        conn.rollback()
        print(f"환불 실패: {e}")


# 회원만 구매이력 조회
def purchase_history(conn, user):
    print("\n── 구매이력 조회 ──")
    rows = conn.execute(
        """SELECT 판매.판매번호, 판매.판매일자, 판매.총판매금액,
                  판매.결제수단, 판매.지점명
           FROM 판매
           WHERE 판매.고객ID=?
           ORDER BY 판매.판매일자 DESC""",
        (user["고객ID"],)
    ).fetchall()

    if not rows:
        print("구매이력이 없습니다.")
        return

    print(f"\n{'판매번호':<12} {'날짜':<12} {'금액':>10} {'결제수단':<8} {'지점명'}")
    print("-" * 60)
    for r in rows:
        print(f"{r['판매번호']:<12} {r['판매일자']:<12} {r['총판매금액']:>10,}원 {r['결제수단']:<8} {r['지점명']}")

    # 상세 조회
    detail_choice = input("\n상세 조회할 판매번호 입력 (Enter=건너뜀): ").strip()
    if detail_choice:
        details = conn.execute(
            """SELECT 상품.이름, 판매상세.수량, 판매상세.판매단가
               FROM 판매상세
               JOIN 상품 ON 판매상세.상품코드=상품.상품코드
               WHERE 판매상세.판매번호=?""",
            (detail_choice,)
        ).fetchall()
        if details:
            print(f"\n[{detail_choice}] 상세내역")
            for d in details:
                print(f"  {d['이름']} x {d['수량']} = {d['수량']*d['판매단가']:,}원")
        else:
            print("해당 판매번호를 찾을 수 없습니다.")

    # 환불
    print("\n── 환불 규정 안내 ──")
    print("· 구매 당일에 한해 환불 가능합니다.")
    print("· 제품 이상(변질, 이물질 등)이 확인된 경우 환불 가능합니다.")
    print("· 단순 변심으로 인한 환불은 제한됩니다.")
    print("· 환불 시 결제 금액 전액이 반환됩니다.")
    refund_choice = input("\n환불할 판매번호 입력 (Enter=건너뜀): ").strip()
    if refund_choice:
        refund(conn, user, refund_choice)


# 로그인 후 메뉴
def member_menu(conn, user):
    branch = select_branch(conn)

    while True:
        print("\n" + "=" * 50)
        print(f"소비자 CLI - {user['이름']}님 ({branch})")
        print("=" * 50)
        print("1. 상품 검색")
        print("2. 구매")
        print("3. 구매이력 조회 / 환불")
        print("4. 매장 변경")
        print("0. 로그아웃")
        print("=" * 50)

        choice = input("메뉴 선택 > ").strip()

        try:
            if choice == "1":
                search_products(conn, branch)
            elif choice == "2":
                purchase(conn, user, branch)
            elif choice == "3":
                if user["회원여부"]:
                    purchase_history(conn, user)
                else:
                    print("회원만 이용 가능합니다.")
            elif choice == "4":
                branch = select_branch(conn)
            elif choice == "0":
                break
            else:
                print("잘못된 메뉴 번호입니다.")
        except sqlite3.Error as e:
            print(f"SQLite 오류: {e}")
        except Exception as e:
            print(f"오류: {e}")


# 메인
def main():
    print("<소비자 CLI>")
    try:
        conn = get_connection()
        print(f"DB 접속 완료: {DB_PATH}")
    except FileNotFoundError as e:
        print(e)
        return

    while True:
        print("\n" + "=" * 50)
        print("파리바게트 소비자 서비스")
        print("=" * 50)
        print("1. 회원 로그인")
        print("2. 비회원으로 구매")
        print("0. 종료")
        print("=" * 50)

        choice = input("메뉴 선택 > ").strip()

        try:
            if choice == "1":
                user = login(conn)
                if user:
                    member_menu(conn, user)
            elif choice == "2":
                user = enter_as_nonmember(conn)
                if user:
                    member_menu(conn, user)
            elif choice == "0":
                break
            else:
                print("잘못된 메뉴 번호입니다.")
        except sqlite3.Error as e:
            print(f"SQLite 오류: {e}")
        except Exception as e:
            print(f"오류: {e}")

    conn.close()
    print("종료합니다.")


if __name__ == "__main__":
    main()

<소비자 CLI>
DB 접속 완료: ../schema/파리바게트.db

파리바게트 소비자 서비스
1. 회원 로그인
2. 비회원으로 구매
0. 종료


메뉴 선택 >  1



── 회원 로그인 ──


아이디:  shwshw2003
비밀번호:  tjgPdnjs00!



환영합니다, 서혜원님!

── 매장 선택 ──

번호   지점명             지역       영업시간
--------------------------------------------------
1    수원점             경기       07:30-21:30
2    판교점             경기       07:00-22:00
3    광주점             광주       08:00-21:00
4    대구점             대구       07:30-22:00
5    대전점             대전       08:00-21:30
6    부산서면점           부산       07:00-22:00
7    부산해운대점          부산       08:00-23:00
8    강남점             서울       07:00-22:00
9    명동점             서울       08:00-22:00
10   신촌점             서울       07:30-22:00
11   이태원점            서울       09:00-23:00
12   잠실점             서울       07:00-22:00
13   홍대점             서울       08:00-23:00
14   인천점             인천       08:00-21:00
15   제주점             제주       08:00-21:00


...


소비자 CLI - 서혜원님 (강남점)
1. 상품 검색
2. 구매
3. 구매이력 조회 / 환불
4. 매장 변경
0. 로그아웃


메뉴 선택 >  0



파리바게트 소비자 서비스
1. 회원 로그인
2. 비회원으로 구매
0. 종료


메뉴 선택 >  0


종료합니다.
